In [1]:
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [2]:
# Loading all crime data and clustering
df_crimes = pd.read_csv("london_all_data_uncleaned.csv").copy()
df_crimes = df_crimes.dropna(subset=["LSOA code"])
df_crimes = df_crimes.dropna(subset=["Crime type"])
df_crimes = df_crimes.dropna(subset=["Month"])
df_crimes = df_crimes.drop(columns=["Context"])

df_crimes["Month"] = pd.to_datetime(df_crimes["Month"])
# df_crimes["Month"] = df_crimes["Month"].dt.month

df_crimes = (df_crimes.groupby(["LSOA code", "Month", "Crime type"]).size().reset_index(name="crime_count"))

In [3]:
# Encode categorical features
le_crime = LabelEncoder()
le_lsoa  = LabelEncoder()

df_crimes["crime_type_enc"] = le_crime.fit_transform(df_crimes["Crime type"].fillna("Unknown"))
df_crimes["lsoa_enc"] = le_lsoa.fit_transform(df_crimes["LSOA code"].fillna("Unknown"))
df_crimes["Month"] = le_lsoa.fit_transform(df_crimes["Month"].fillna("Unknown"))

df = df_crimes.sort_values(["Month"])
split = int(len(df) * 0.666666)
train = df.iloc[:split]
test = df.iloc[split:]


features = ["crime_type_enc", "Month", "lsoa_enc"]
target   = "crime_count" 

# X = df_crimes[features]
# y = df_crimes["crime_count"]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = train[features]
y_train = train["crime_count"]

X_test = test[features]
y_test = test["crime_count"]

model = xgb.XGBRegressor(
    objective="count:poisson",
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=10,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.5,
    gamma=0.1,
    random_state=42
)
model.fit(X_train, y_train)

preds = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"RMSE: {rmse:.2f}")

RMSE: 5.55


In [4]:
df_crimes["crime_count"].mean()

3.0779455169112953